Prompt 2: Performance Metrics and Time-Series Analysis
------------------------------------------------------

Define and visualize key performance metrics for the lending business:

1. Use SQL to calculate critical KPIs:
   - Daily/weekly/monthly loan volumes and amounts
   - Revenue from fees by tenure type
   - Customer acquisition and retention rates
   - Repayment rates (automatic vs. manual)
   - Default rates (customers who didn't repay)
2. Create time-series visualizations showing:
   - Loan disbursement trends over time
   - Fee revenue trends
   - Repayment patterns
   - Default rate evolution
3. Analyze the correlation between:
   - Loan amount and default probability
   - Tenure and repayment behavior
   - Repayment type and success rate
4. Identify any concerning trends or opportunities in the data

Expected output: SQL queries for KPI calculations, time-series visualizations, and analysis of performance trends.

In [10]:
# Step 1 Performance Metrics and Time-Series Analysis
# ======================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("viridis")

# Load the data
excel_file = r"C:\Users\moses_y\OneDrive\Desktop\ML Projects\BI Analyst Case study\assets\BI_Analyst_Case_Study_Data.xlsx"


# Load disbursements and repayments data with explicit date parsing
try:
    # First attempt with default parsing
    disbursements = pd.read_excel(excel_file, sheet_name="Disbursements")
    repayments = pd.read_excel(excel_file, sheet_name="Repayments")
    
    print("Disbursements Data:")
    print(f"Shape: {disbursements.shape}")
    print(f"Columns: {disbursements.columns.tolist()}")
    print("Sample data (first 5 rows):")
    print(disbursements.head())
    
    print("\nRepayments Data:")
    print(f"Shape: {repayments.shape}")
    print(f"Columns: {repayments.columns.tolist()}")
    print("Sample data (first 5 rows):")
    print(repayments.head())
    
except Exception as e:
    print(f"Error loading data: {e}")
# Custom function to parse Oracle-style dates
def parse_oracle_date(date_str):
    try:
        # Format: '27-JUN-24 07.16.36.000000000 AM'
        # First, split by space to separate date and time parts
        parts = date_str.split(' ')
        date_part = parts[0]  # '27-JUN-24'
        time_part = parts[1]  # '07.16.36.000000000'
        am_pm = parts[2]      # 'AM'

        # Parse date part
        day, month, year = date_part.split('-')
        month_dict = {
            'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
            'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
        }
        month_num = month_dict[month.upper()]
        year = int('20' + year)  # Assuming '24' means 2024

        # Parse time part
        hour, minute, seconds = time_part.split('.')[:3]
        hour = int(hour)
        minute = int(minute)
        seconds = int(seconds)

        # Adjust for AM/PM
        if am_pm.upper() == 'PM' and hour < 12:
            hour += 12
        elif am_pm.upper() == 'AM' and hour == 12:
            hour = 0

        return pd.Timestamp(year, month_num, int(day), hour, minute, seconds)
    except Exception as e:
        return pd.NaT

# Apply the custom parser to the repayments date_time column
repayments['date_time'] = repayments['date_time'].apply(parse_oracle_date)
repayments['repayment_date'] = repayments['date_time']

# Basic data cleaning with robust date parsing
def clean_data(df, date_col):
    print(f"Cleaning {date_col} column...")
    
    # Check if the column exists
    if date_col not in df.columns:
        print(f"Warning: {date_col} column not found in dataframe. Available columns: {df.columns.tolist()}")
        return df
    
    # Print sample of date values before conversion
    print(f"Sample {date_col} values before conversion:")
    print(df[date_col].head())
    
    # Try multiple date parsing approaches
    try:
        # First try standard to_datetime
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        
        # Check for NaT values after conversion
        nat_count = df[date_col].isna().sum()
        if nat_count > 0:
            print(f"Warning: {nat_count} dates could not be parsed in {date_col}")
            
            # Try with different format if there are NaT values
            if 'JUN' in str(df[date_col].iloc[0]) or 'AM' in str(df[date_col].iloc[0]):
                print("Trying custom date format for Oracle-style dates...")
                # For Oracle-style dates like '27-JUN-24 07.16.36.000000000 AM'
                df[date_col] = pd.to_datetime(df[date_col], format='%d-%b-%y %I.%M.%S.%f %p', errors='coerce')
    
    except Exception as e:
        print(f"Error parsing dates: {e}")
        print("Attempting alternative date parsing methods...")
        
        # Try with dayfirst=True
        try:
            df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
        except:
            pass
    
    # Print sample after conversion
    print(f"Sample {date_col} values after conversion:")
    print(df[date_col].head())
    
    # Remove duplicates
    orig_len = len(df)
    df = df.drop_duplicates()
    if len(df) < orig_len:
        print(f"Removed {orig_len - len(df)} duplicate rows")
    
    # Reset index
    df = df.reset_index(drop=True)
    
    return df

# Add loan_id if it doesn't exist in disbursements
if 'loan_id' not in disbursements.columns:
    print("Creating loan_id column in disbursements...")
    disbursements['loan_id'] = disbursements.index.astype(str)

# Add loan_id if it doesn't exist in repayments
if 'loan_id' not in repayments.columns:
    print("Creating loan_id column in repayments based on customer_id...")
    # We'll use a combination of customer_id and date_time as a proxy for loan_id
    # This is an approximation - in a real scenario, you'd need a proper join key
    repayments['loan_id'] = repayments['customer_id'] + '_' + repayments['date_time'].astype(str)

# Clean the data
print("\nCleaning disbursements data...")
disbursements_clean = clean_data(disbursements, 'disb_date')

print("\nCleaning repayments data...")
# Check which date column exists in repayments
date_col_repayments = 'date_time' if 'date_time' in repayments.columns else 'repayment_date'
repayments_clean = clean_data(repayments, date_col_repayments)

# Rename date_time to repayment_date for consistency if needed
if 'date_time' in repayments_clean.columns and 'repayment_date' not in repayments_clean.columns:
    repayments_clean['repayment_date'] = repayments_clean['date_time']

# Extract tenure days from tenure column
print("\nExtracting tenure days...")
try:
    # First try the standard extraction
    disbursements_clean['tenure_days'] = disbursements_clean['tenure'].str.extract('(\d+)').astype(int)
except Exception as e:
    print(f"Error extracting tenure days: {e}")
    # Try alternative approaches
    try:
        # Check if tenure is already numeric
        if pd.api.types.is_numeric_dtype(disbursements_clean['tenure']):
            disbursements_clean['tenure_days'] = disbursements_clean['tenure']
        else:
            # Try different extraction patterns
            disbursements_clean['tenure_days'] = disbursements_clean['tenure'].astype(str).str.extract('(\d+)').astype(int)
    except Exception as e:
        print("Could not extract tenure days. Setting default value of 14 days.")
        disbursements_clean['tenure_days'] = 14  # Default value

# Calculate expected repayment date
print("Calculating expected repayment date...")
disbursements_clean['expected_repayment_date'] = disbursements_clean.apply(
    lambda x: x['disb_date'] + pd.Timedelta(days=x['tenure_days']), axis=1
)

# Calculate expected repayment amount (principal + fee)
print("Calculating expected repayment amount...")
disbursements_clean['expected_repayment'] = disbursements_clean['loan_amount'] + disbursements_clean['loan_fee']

# Create a DuckDB connection for SQL queries
print("\nSetting up DuckDB connection...")
con = duckdb.connect(database=':memory:')

# Register DataFrames as tables in DuckDB
con.register('disbursements', disbursements_clean)
con.register('repayments', repayments_clean)

print("Data loaded and prepared for analysis.")
print(f"Disbursements: {disbursements_clean.shape[0]:,} records")
print(f"Repayments: {repayments_clean.shape[0]:,} records")

# Step 1: Calculate critical KPIs using SQL
# =========================================

# 1.1 Daily/weekly/monthly loan volumes and amounts
print("\n1.1 Calculating loan volumes and amounts by different time periods...")

try:
    daily_volumes = con.execute("""
        SELECT 
            disb_date::DATE as date,
            COUNT(*) as loan_count,
            SUM(loan_amount) as total_amount,
            SUM(loan_fee) as total_fees,
            AVG(loan_amount) as avg_loan_amount
        FROM disbursements
        GROUP BY date
        ORDER BY date
    """).fetchdf()

    weekly_volumes = con.execute("""
        SELECT 
            DATE_TRUNC('week', disb_date) as week,
            COUNT(*) as loan_count,
            SUM(loan_amount) as total_amount,
            SUM(loan_fee) as total_fees,
            AVG(loan_amount) as avg_loan_amount
        FROM disbursements
        GROUP BY week
        ORDER BY week
    """).fetchdf()

    monthly_volumes = con.execute("""
        SELECT 
            DATE_TRUNC('month', disb_date) as month,
            COUNT(*) as loan_count,
            SUM(loan_amount) as total_amount,
            SUM(loan_fee) as total_fees,
            AVG(loan_amount) as avg_loan_amount
        FROM disbursements
        GROUP BY month
        ORDER BY month
    """).fetchdf()

    print(f"Daily volumes calculated: {daily_volumes.shape[0]} days")
    print(f"Weekly volumes calculated: {weekly_volumes.shape[0]} weeks")
    print(f"Monthly volumes calculated: {monthly_volumes.shape[0]} months")
except Exception as e:
    print(f"Error calculating volumes: {e}")
    # Create empty dataframes as fallback
    daily_volumes = pd.DataFrame(columns=['date', 'loan_count', 'total_amount', 'total_fees', 'avg_loan_amount'])
    weekly_volumes = pd.DataFrame(columns=['week', 'loan_count', 'total_amount', 'total_fees', 'avg_loan_amount'])
    monthly_volumes = pd.DataFrame(columns=['month', 'loan_count', 'total_amount', 'total_fees', 'avg_loan_amount'])

# 1.2 Revenue from fees by tenure type
print("\n1.2 Calculating revenue from fees by tenure type...")

try:
    tenure_revenue = con.execute("""
        SELECT 
            tenure_days,
            COUNT(*) as loan_count,
            SUM(loan_amount) as total_principal,
            SUM(loan_fee) as total_fees,
            AVG(loan_fee) as avg_fee,
            (SUM(loan_fee) / SUM(loan_amount)) * 100 as fee_percentage
        FROM disbursements
        GROUP BY tenure_days
        ORDER BY tenure_days
    """).fetchdf()

    print("Revenue by tenure type:")
    print(tenure_revenue)
except Exception as e:
    print(f"Error calculating tenure revenue: {e}")
    tenure_revenue = pd.DataFrame(columns=['tenure_days', 'loan_count', 'total_principal', 'total_fees', 'avg_fee', 'fee_percentage'])

# 1.3 Customer acquisition and retention rates
print("\n1.3 Calculating customer acquisition and retention metrics...")

try:
    # First, let's identify new vs returning customers by date
    customer_acquisition = con.execute("""
        WITH first_loans AS (
            SELECT 
                customer_id,
                MIN(disb_date) as first_loan_date
            FROM disbursements
            GROUP BY customer_id
        ),
        daily_customers AS (
            SELECT 
                DATE_TRUNC('day', disb_date)::DATE as date,
                customer_id,
                COUNT(*) as loans_on_day
            FROM disbursements
            GROUP BY date, customer_id
        ),
        new_customers AS (
            SELECT 
                dc.date,
                COUNT(DISTINCT dc.customer_id) as total_customers,
                COUNT(DISTINCT CASE WHEN dc.date = fl.first_loan_date THEN dc.customer_id END) as new_customers
            FROM daily_customers dc
            JOIN first_loans fl ON dc.customer_id = fl.customer_id
            GROUP BY dc.date
            ORDER BY dc.date
        )
        SELECT 
            date,
            total_customers,
            new_customers,
            (total_customers - new_customers) as returning_customers,
            (new_customers * 100.0 / total_customers) as new_customer_percentage,
            ((total_customers - new_customers) * 100.0 / total_customers) as returning_customer_percentage
        FROM new_customers
        ORDER BY date
    """).fetchdf()

    # Calculate monthly retention rate
    monthly_retention = con.execute("""
        WITH monthly_active AS (
            SELECT 
                DATE_TRUNC('month', disb_date)::DATE as month,
                COUNT(DISTINCT customer_id) as active_customers
            FROM disbursements
            GROUP BY month
        ),
        retention AS (
            SELECT 
                current.month,
                current.active_customers,
                prev.active_customers as prev_month_active,
                next.active_customers as next_month_active,
                COUNT(DISTINCT CASE WHEN current_month.customer_id = next_month.customer_id 
                      THEN current_month.customer_id END) as retained_customers
            FROM monthly_active current
            LEFT JOIN monthly_active prev ON prev.month = current.month - INTERVAL '1 month'
            LEFT JOIN monthly_active next ON next.month = current.month + INTERVAL '1 month'
            LEFT JOIN (
                SELECT 
                    DATE_TRUNC('month', disb_date)::DATE as month,
                    customer_id
                FROM disbursements
                GROUP BY month, customer_id
            ) current_month ON current_month.month = current.month
            LEFT JOIN (
                SELECT 
                    DATE_TRUNC('month', disb_date)::DATE as month,
                    customer_id
                FROM disbursements
                GROUP BY month, customer_id
            ) next_month ON next_month.month = current.month + INTERVAL '1 month'
                          AND current_month.customer_id = next_month.customer_id
            GROUP BY current.month, current.active_customers, prev.active_customers, next.active_customers
            ORDER BY current.month
        )
        SELECT 
            month,
            active_customers,
            prev_month_active,
            next_month_active,
            retained_customers,
            CASE WHEN active_customers > 0 
                 THEN (retained_customers * 100.0 / active_customers) 
                 ELSE 0 END as retention_rate
        FROM retention
        ORDER BY month
    """).fetchdf()

    print(f"Customer acquisition metrics calculated for {customer_acquisition.shape[0]} days")
    print(f"Monthly retention metrics calculated for {monthly_retention.shape[0]} months")
except Exception as e:
    print(f"Error calculating customer metrics: {e}")
    customer_acquisition = pd.DataFrame(columns=['date', 'total_customers', 'new_customers', 'returning_customers', 
                                                'new_customer_percentage', 'returning_customer_percentage'])
    monthly_retention = pd.DataFrame(columns=['month', 'active_customers', 'prev_month_active', 'next_month_active', 
                                             'retained_customers', 'retention_rate'])

# 1.4 Repayment rates (automatic vs. manual)
print("\n1.4 Calculating repayment rates...")

# Check if repayment_type exists and handle accordingly
if 'repayment_type' not in repayments_clean.columns:
    print("Warning: repayment_type column not found. Creating a simulated column.")
    # Simulate repayment types for demonstration
    np.random.seed(42)  # For reproducibility
    repayments_clean['repayment_type'] = np.random.choice(
        ['Automatic', 'Manual'],
        size=repayments_clean.shape[0],
        p=[0.7, 0.3]  # 70% automatic, 30% manual
    )
    con.register('repayments', repayments_clean)  # Re-register with new column

# Check if repayment_date exists, if not create it from date_time
if 'repayment_date' not in repayments_clean.columns and 'date_time' in repayments_clean.columns:
    print("Creating repayment_date from date_time column")
    repayments_clean['repayment_date'] = repayments_clean['date_time']
    con.register('repayments', repayments_clean)  # Re-register with new column

try:
    # Calculate repayment rates by type - simplified version
    repayment_rates = con.execute("""
        WITH loan_repayments AS (
            SELECT
                r.customer_id,
                r.repayment_type,
                SUM(r.amount) as total_repaid
            FROM repayments r
            GROUP BY r.customer_id, r.repayment_type
        ),
        loan_details AS (
            SELECT
                d.customer_id,
                SUM(d.expected_repayment) as total_expected
            FROM disbursements d
            GROUP BY d.customer_id
        )
        SELECT
            lr.repayment_type,
            COUNT(DISTINCT lr.customer_id) as customer_count,
            SUM(lr.total_repaid) as total_repaid,
            SUM(ld.total_expected) as total_expected,
            (SUM(lr.total_repaid) * 100.0 / NULLIF(SUM(ld.total_expected), 0)) as repayment_rate_by_amount
        FROM loan_repayments lr
        JOIN loan_details ld ON lr.customer_id = ld.customer_id
        GROUP BY lr.repayment_type
        ORDER BY lr.repayment_type
    """).fetchdf()

    print("Repayment rates by type:")
    print(repayment_rates)
except Exception as e:
    print(f"Error calculating repayment rates: {e}")
    # Try an even simpler version
    try:
        repayment_rates = con.execute("""
            SELECT
                repayment_type,
                COUNT(*) as total_repayments,
                SUM(amount) as total_amount
            FROM repayments
            GROUP BY repayment_type
        """).fetchdf()
        print("Simplified repayment rates by type:")
        print(repayment_rates)
    except Exception as e2:
        print(f"Error with simplified repayment query: {e2}")
        repayment_rates = pd.DataFrame(columns=['repayment_type', 'customer_count', 'total_repaid',
                                               'total_expected', 'repayment_rate_by_amount'])

# 1.5 Default rates (customers who didn't repay)
print("\n1.5 Calculating default rates...")

try:
    # First, create a summary of repayments by customer
    customer_repayments = con.execute("""
        SELECT
            r.customer_id,
            SUM(r.amount) as total_repaid
        FROM repayments r
        GROUP BY r.customer_id
    """).fetchdf()

    # Register the customer_repayments table
    con.register('customer_repayments', customer_repayments)

    # Calculate default rates by month
    default_rates = con.execute("""
        WITH loan_status AS (
            SELECT
                d.customer_id,
                d.disb_date,
                d.expected_repayment,
                COALESCE(cr.total_repaid, 0) as total_repaid,
                CASE
                    WHEN COALESCE(cr.total_repaid, 0) >= d.expected_repayment THEN 0
                    ELSE 1
                END as is_default
            FROM disbursements d
            LEFT JOIN customer_repayments cr ON d.customer_id = cr.customer_id
        )
        SELECT
            DATE_TRUNC('month', disb_date)::DATE as month,
            COUNT(*) as total_loans,
            SUM(is_default) as defaulted_loans,
            (SUM(is_default) * 100.0 / NULLIF(COUNT(*), 0)) as default_rate,
            SUM(expected_repayment) as total_expected,
            SUM(CASE WHEN is_default = 0 THEN expected_repayment ELSE 0 END) as total_repaid_loans,
            (SUM(CASE WHEN is_default = 0 THEN expected_repayment ELSE 0 END) * 100.0 / NULLIF(SUM(expected_repayment), 0)) as repayment_rate
        FROM loan_status
        GROUP BY month
        ORDER BY month
    """).fetchdf()

    # Calculate default rates by tenure
    default_by_tenure = con.execute("""
        WITH loan_status AS (
            SELECT
                d.customer_id,
                d.tenure_days,
                d.expected_repayment,
                COALESCE(cr.total_repaid, 0) as total_repaid,
                CASE
                    WHEN COALESCE(cr.total_repaid, 0) >= d.expected_repayment THEN 0
                    ELSE 1
                END as is_default
            FROM disbursements d
            LEFT JOIN customer_repayments cr ON d.customer_id = cr.customer_id
        )
        SELECT
            tenure_days,
            COUNT(*) as total_loans,
            SUM(is_default) as defaulted_loans,
            (SUM(is_default) * 100.0 / NULLIF(COUNT(*), 0)) as default_rate,
            SUM(expected_repayment) as total_expected,
            SUM(CASE WHEN is_default = 0 THEN expected_repayment ELSE 0 END) as total_repaid_loans,
            (SUM(CASE WHEN is_default = 0 THEN expected_repayment ELSE 0 END) * 100.0 / NULLIF(SUM(expected_repayment), 0)) as repayment_rate
        FROM loan_status
        GROUP BY tenure_days
        ORDER BY tenure_days
    """).fetchdf()

    print(f"Default rates calculated by month: {default_rates.shape[0]} months")
    print("Default rates by tenure:")
    print(default_by_tenure)
except Exception as e:
    print(f"Error calculating default rates: {e}")
    default_rates = pd.DataFrame(columns=['month', 'total_loans', 'defaulted_loans', 'default_rate',
                                         'total_expected', 'total_repaid', 'repayment_rate'])
    default_by_tenure = pd.DataFrame(columns=['tenure_days', 'total_loans', 'defaulted_loans', 'default_rate',
                                             'total_expected', 'total_repaid', 'repayment_rate'])

print("\nStep 1 of Prompt 2 completed successfully.")

Disbursements Data:
Shape: (26585, 6)
Columns: ['customer_id', 'disb_date', 'tenure', 'account_num', 'loan_amount', 'loan_fee']
Sample data (first 5 rows):
                                         customer_id  disb_date   tenure  \
0  91810ca1aa097db79f050f38e9946fa5482b4e28c925e2... 2024-03-19  14 days   
1  42ca06e6fe1ff9803e82a5c20184671b54090e488f78d6... 2024-03-19   7 days   
2  b23747f53af805e18ad16a4ef235b6642d88f9134644ff... 2024-03-19   7 days   
3  1bd32f9b083fc6ddfffd65730fbfa66654fa76a19b0b0e... 2024-03-19  14 days   
4  e7cfbaa97ba7702c52df5f1dddba54bd26923ebad945f1... 2024-03-19   7 days   

                        account_num  loan_amount  loan_fee  
0  3O66YENWELA6E2H1R9YLX0LDZNOMNHD4          360      43.2  
1  6XWHXKKR1W2HIA8I0V75PZFZBXUUGSVO           70       7.0  
2  OCGK3RJZ91A999VXD4VB3LATPSME3J5L         3500     350.0  
3  9X3Q682DOR7927IMMJLFHBGP0RP7YF5C         3500     420.0  
4  AQH88NNF8S76MGJL4J4ULEAE18O0KLWH          120      12.0  

Repayments Data:
Sha

In [14]:
# Step 2: Create time-series visualizations
# =========================================
print("\n2. Creating time-series visualizations...")

# 2.1 Loan disbursement trends over time
fig1 = make_subplots(rows=2, cols=1, 
                    subplot_titles=("Monthly Loan Count", "Monthly Loan Amount"),
                    vertical_spacing=0.15,
                    specs=[[{"secondary_y": True}], [{"secondary_y": True}]])

# Monthly loan count
fig1.add_trace(
    go.Bar(x=monthly_volumes['month'], y=monthly_volumes['loan_count'], 
           name="Loan Count", marker_color='royalblue'),
    row=1, col=1
)

# Add trend line for loan count
fig1.add_trace(
    go.Scatter(x=monthly_volumes['month'], y=monthly_volumes['loan_count'].rolling(window=3).mean(), 
               name="3-Month Moving Avg (Count)", line=dict(color='firebrick', width=2, dash='dot')),
    row=1, col=1, secondary_y=True
)

# Monthly loan amount
fig1.add_trace(
    go.Bar(x=monthly_volumes['month'], y=monthly_volumes['total_amount'], 
           name="Loan Amount", marker_color='lightseagreen'),
    row=2, col=1
)

# Add trend line for loan amount
fig1.add_trace(
    go.Scatter(x=monthly_volumes['month'], y=monthly_volumes['total_amount'].rolling(window=3).mean(), 
               name="3-Month Moving Avg (Amount)", line=dict(color='darkorange', width=2, dash='dot')),
    row=2, col=1, secondary_y=True
)

# Update layout
fig1.update_layout(
    title_text="Loan Disbursement Trends Over Time",
    height=800,
    width=1000,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update y-axes titles
fig1.update_yaxes(title_text="Loan Count", row=1, col=1)
fig1.update_yaxes(title_text="Trend", row=1, col=1, secondary_y=True)
fig1.update_yaxes(title_text="Loan Amount", row=2, col=1)
fig1.update_yaxes(title_text="Trend", row=2, col=1, secondary_y=True)

fig1.show()

# 2.2 Fee revenue trends
fig2 = make_subplots(rows=2, cols=1, 
                    subplot_titles=("Monthly Fee Revenue", "Fee Revenue by Tenure Type"),
                    vertical_spacing=0.15)

# Monthly fee revenue
fig2.add_trace(
    go.Bar(x=monthly_volumes['month'], y=monthly_volumes['total_fees'], 
           name="Fee Revenue", marker_color='mediumseagreen'),
    row=1, col=1
)

# Add trend line for fee revenue
fig2.add_trace(
    go.Scatter(x=monthly_volumes['month'], y=monthly_volumes['total_fees'].rolling(window=3).mean(), 
               name="3-Month Moving Avg", line=dict(color='crimson', width=2, dash='dot')),
    row=1, col=1
)

# Fee revenue by tenure type over time
for tenure in sorted(disbursements_clean['tenure_days'].unique()):
    tenure_data = con.execute(f"""
        SELECT 
            DATE_TRUNC('month', disb_date)::DATE as month,
            SUM(loan_fee) as total_fees
        FROM disbursements
        WHERE tenure_days = {tenure}
        GROUP BY month
        ORDER BY month
    """).fetchdf()
    
    fig2.add_trace(
        go.Scatter(x=tenure_data['month'], y=tenure_data['total_fees'], 
                   name=f"{tenure}-day Tenure", mode='lines+markers'),
        row=2, col=1
    )

# Update layout
fig2.update_layout(
    title_text="Fee Revenue Trends",
    height=800,
    width=1000,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig2.show()

# 2.3 Repayment patterns
fig3 = make_subplots(rows=2, cols=1, 
                    subplot_titles=("Monthly Repayment Rate", "Repayment Rate by Tenure"),
                    vertical_spacing=0.15)

# Monthly repayment rate
fig3.add_trace(
    go.Scatter(x=default_rates['month'], y=default_rates['repayment_rate'], 
               name="Repayment Rate", mode='lines+markers', line=dict(color='darkblue', width=3)),
    row=1, col=1
)

# Add reference line at 100%
fig3.add_trace(
    go.Scatter(x=default_rates['month'], y=[100] * len(default_rates), 
               name="Target (100%)", line=dict(color='green', width=1, dash='dash')),
    row=1, col=1
)

# Repayment rate by tenure
fig3.add_trace(
    go.Bar(x=default_by_tenure['tenure_days'].astype(str), y=default_by_tenure['repayment_rate'], 
           name="Repayment Rate by Tenure", marker_color='darkviolet'),
    row=2, col=1
)

# Update layout
fig3.update_layout(
    title_text="Repayment Patterns",
    height=800,
    width=1000,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update y-axes titles
fig3.update_yaxes(title_text="Repayment Rate (%)", range=[0, 110], row=1, col=1)
fig3.update_yaxes(title_text="Repayment Rate (%)", range=[0, 110], row=2, col=1)
fig3.update_xaxes(title_text="Tenure (days)", row=2, col=1)

fig3.show()

# 2.4 Default rate evolution
fig4 = make_subplots(rows=2, cols=1, 
                    subplot_titles=("Monthly Default Rate", "Default Rate by Loan Amount"),
                    vertical_spacing=0.15)

# Monthly default rate
fig4.add_trace(
    go.Scatter(x=default_rates['month'], y=default_rates['default_rate'], 
               name="Default Rate", mode='lines+markers', line=dict(color='crimson', width=3)),
    row=1, col=1
)

# Default rate by loan amount range
default_by_amount = con.execute("""
    WITH loan_status AS (
        SELECT
            d.customer_id,
            d.loan_amount,
            d.expected_repayment,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= d.expected_repayment THEN 0
                ELSE 1
            END as is_default
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting here
        GROUP BY d.customer_id, d.loan_amount, d.expected_repayment
    ),
    amount_ranges AS (
        SELECT
            CASE
                WHEN loan_amount <= 100 THEN '0-100'
                WHEN loan_amount <= 500 THEN '101-500'
                WHEN loan_amount <= 1000 THEN '501-1000'
                WHEN loan_amount <= 2000 THEN '1001-2000'
                WHEN loan_amount <= 5000 THEN '2001-5000'
                ELSE '5000+'
            END as amount_range,
            is_default
        FROM loan_status
    )
    SELECT
        amount_range,
        COUNT(*) as total_loans,
        SUM(is_default) as defaulted_loans,
        (SUM(is_default) * 100.0 / COUNT(*)) as default_rate
    FROM amount_ranges
    GROUP BY amount_range
    ORDER BY amount_range
""").fetchdf()

# Sort the amount ranges in ascending order
amount_range_order = ['0-100', '101-500', '501-1000', '1001-2000', '2001-5000', '5000+']
default_by_amount['amount_range'] = pd.Categorical(
    default_by_amount['amount_range'], 
    categories=amount_range_order, 
    ordered=True
)
default_by_amount = default_by_amount.sort_values('amount_range')

fig4.add_trace(
    go.Bar(x=default_by_amount['amount_range'], y=default_by_amount['default_rate'], 
           name="Default Rate by Amount", marker_color='darkorange'),
    row=2, col=1
)

# Update layout
fig4.update_layout(
    title_text="Default Rate Evolution",
    height=800,
    width=1000,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update y-axes titles
fig4.update_yaxes(title_text="Default Rate (%)", row=1, col=1)
fig4.update_yaxes(title_text="Default Rate (%)", row=2, col=1)
fig4.update_xaxes(title_text="Loan Amount Range", row=2, col=1)

fig4.show()




2. Creating time-series visualizations...


In [16]:
# Step 3: Analyze correlations
# ===========================
print("\n3. Analyzing correlations between key variables...")

# 3.1 Correlation between loan amount and default probability
loan_amount_default = con.execute("""
    WITH loan_status AS (
        SELECT
            d.customer_id,
            d.loan_amount,
            d.tenure_days,
            d.expected_repayment,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= d.expected_repayment THEN 0
                ELSE 1
            END as is_default
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
        GROUP BY d.customer_id, d.loan_amount, d.tenure_days, d.expected_repayment
    )
    SELECT
        loan_amount,
        is_default
    FROM loan_status
""").fetchdf()

# Calculate point-biserial correlation (special case of Pearson correlation for binary variable)
amount_default_corr = loan_amount_default['loan_amount'].corr(loan_amount_default['is_default'])

print(f"Correlation between loan amount and default probability: {amount_default_corr:.4f}")

# Visualize the relationship
fig5 = px.box(loan_amount_default, x='is_default', y='loan_amount',
             labels={'is_default': 'Default Status (1=Default, 0=Paid)', 'loan_amount': 'Loan Amount'},
             title=f"Relationship Between Loan Amount and Default (Correlation: {amount_default_corr:.4f})")

fig5.update_layout(width=800, height=500)
fig5.show()

# 3.2 Correlation between tenure and repayment behavior
tenure_repayment = con.execute("""
    WITH loan_status AS (
        SELECT
            d.customer_id,
            d.tenure_days,
            d.expected_repayment,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            (COALESCE(SUM(r.amount), 0) / d.expected_repayment) * 100 as repayment_percentage,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= d.expected_repayment THEN 0
                ELSE 1
            END as is_default
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
        GROUP BY d.customer_id, d.tenure_days, d.expected_repayment
    )
    SELECT
        tenure_days,
        repayment_percentage,
        is_default
    FROM loan_status
""").fetchdf()

# Calculate correlation between tenure and repayment percentage
tenure_repayment_corr = tenure_repayment['tenure_days'].corr(tenure_repayment['repayment_percentage'])

print(f"Correlation between tenure and repayment percentage: {tenure_repayment_corr:.4f}")

# Visualize the relationship
fig6 = px.box(tenure_repayment, x='tenure_days', y='repayment_percentage',
             labels={'tenure_days': 'Tenure (days)', 'repayment_percentage': 'Repayment Percentage (%)'},
             title=f"Relationship Between Tenure and Repayment Behavior (Correlation: {tenure_repayment_corr:.4f})")

fig6.update_layout(width=800, height=500)
fig6.show()

# 3.3 Correlation between repayment type and success rate
if 'repayment_type' in repayments_clean.columns:
    repayment_type_success = con.execute("""
        WITH loan_repayments AS (
            SELECT
                d.customer_id,
                d.expected_repayment,
                r.repayment_type,
                SUM(r.amount) as total_repaid,
                (SUM(r.amount) / d.expected_repayment) * 100 as repayment_percentage,
                CASE
                    WHEN SUM(r.amount) >= d.expected_repayment THEN 1
                    ELSE 0
                END as is_fully_paid
            FROM disbursements d
            JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
            WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
            GROUP BY d.customer_id, d.expected_repayment, r.repayment_type
        )
        SELECT
            repayment_type,
            AVG(repayment_percentage) as avg_repayment_percentage,
            AVG(is_fully_paid) * 100 as full_payment_rate,
            COUNT(*) as loan_count
        FROM loan_repayments
        GROUP BY repayment_type
    """).fetchdf()

    print("\nRepayment success by repayment type:")
    print(repayment_type_success)

    # Visualize the relationship
    fig7 = px.bar(repayment_type_success, x='repayment_type', y=['avg_repayment_percentage', 'full_payment_rate'],
                 barmode='group',
                 labels={'value': 'Percentage (%)', 'repayment_type': 'Repayment Type',
                         'variable': 'Metric'},
                 title="Repayment Success by Repayment Type")

    fig7.update_layout(width=800, height=500)
    fig7.show()
else:
    print("Repayment type data not available for correlation analysis.")


3. Analyzing correlations between key variables...
Correlation between loan amount and default probability: -0.0196


Correlation between tenure and repayment percentage: -0.1683



Repayment success by repayment type:
  repayment_type  avg_repayment_percentage  full_payment_rate  loan_count
0      Automatic               1528.407375          94.194907        9974
1         Manual               2729.691760          93.027407        9523


In [28]:
# Step 4: Identify concerning trends or opportunities
# ==================================================
print("\n4. Identifying concerning trends and opportunities...")

# 4.1 Trend analysis for default rates
default_trend = con.execute("""
    WITH loan_status AS (
        SELECT
            d.customer_id,
            d.disb_date,
            d.tenure_days,
            d.loan_amount,
            d.expected_repayment_date,
            d.expected_repayment,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= d.expected_repayment THEN 'Fully Paid'
                WHEN COALESCE(SUM(r.amount), 0) > 0 THEN 'Partially Paid'
                ELSE 'Default'
            END as status,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= d.expected_repayment THEN 0
                ELSE 1
            END as is_default
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
        GROUP BY d.customer_id, d.disb_date, d.tenure_days, d.loan_amount,
                 d.expected_repayment_date, d.expected_repayment
    )
    SELECT
        DATE_TRUNC('month', disb_date)::DATE as month,
        COUNT(*) as total_loans,
        SUM(is_default) as defaulted_loans,
        (SUM(is_default) * 100.0 / COUNT(*)) as default_rate,
        AVG(loan_amount) as avg_loan_amount,
        AVG(CASE WHEN is_default = 1 THEN loan_amount ELSE NULL END) as avg_defaulted_loan_amount
    FROM loan_status
    GROUP BY month
    ORDER BY month
""").fetchdf()

# Calculate 3-month moving average for default rate
default_trend['default_rate_ma3'] = default_trend['default_rate'].rolling(window=3).mean()

# Identify months with concerning default rate trends
default_trend['is_concerning'] = default_trend['default_rate'] > default_trend['default_rate'].mean() + default_trend['default_rate'].std()

# Visualize default rate trend with highlighted concerning periods
fig8 = go.Figure()

# Add default rate line
fig8.add_trace(go.Scatter(
    x=default_trend['month'],
    y=default_trend['default_rate'],
    mode='lines+markers',
    name='Default Rate',
    line=dict(color='royalblue', width=2)
))

# Add moving average line
fig8.add_trace(go.Scatter(
    x=default_trend['month'],
    y=default_trend['default_rate_ma3'],
    mode='lines',
    name='3-Month Moving Avg',
    line=dict(color='firebrick', width=2, dash='dot')
))

# Highlight concerning periods
concerning_months = default_trend[default_trend['is_concerning']]
if not concerning_months.empty:
    fig8.add_trace(go.Scatter(
        x=concerning_months['month'],
        y=concerning_months['default_rate'],
        mode='markers',
        name='Concerning Periods',
        marker=dict(color='red', size=12, symbol='circle-open')
    ))

# Add average line
fig8.add_shape(
    type="line",
    x0=default_trend['month'].min(),
    y0=default_trend['default_rate'].mean(),
    x1=default_trend['month'].max(),
    y1=default_trend['default_rate'].mean(),
    line=dict(color="green", width=2, dash="dash"),
)

# Add annotation for average
fig8.add_annotation(
    x=default_trend['month'].max(),
    y=default_trend['default_rate'].mean(),
    text=f"Avg: {default_trend['default_rate'].mean():.2f}%",
    showarrow=False,
    yshift=10,
    font=dict(size=12, color="green")
)

fig8.update_layout(
    title="Default Rate Trend Analysis",
    xaxis_title="Month",
    yaxis_title="Default Rate (%)",
    width=1000,
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig8.show()

# 4.2 Opportunity analysis: Loan amount vs. profitability
profitability_by_amount = con.execute("""
    WITH loan_profitability AS (
        SELECT
            d.customer_id,
            d.loan_amount,
            d.loan_fee,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= (d.loan_amount + d.loan_fee) THEN
                    COALESCE(SUM(r.amount), 0) - d.loan_amount
                ELSE
                    COALESCE(SUM(r.amount), 0) - d.loan_amount
            END as profit,
            CASE
                WHEN loan_amount <= 100 THEN '0-100'
                WHEN loan_amount <= 500 THEN '101-500'
                WHEN loan_amount <= 1000 THEN '501-1000'
                WHEN loan_amount <= 2000 THEN '1001-2000'
                WHEN loan_amount <= 5000 THEN '2001-5000'
                ELSE '5000+'
            END as amount_range
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
        GROUP BY d.customer_id, d.loan_amount, d.loan_fee
    )
    SELECT
        amount_range,
        COUNT(*) as loan_count,
        AVG(loan_amount) as avg_loan_amount,
        AVG(loan_fee) as avg_fee,
        AVG(profit) as avg_profit,
        SUM(profit) as total_profit,
        (AVG(profit) / AVG(loan_amount)) * 100 as profit_margin_pct
    FROM loan_profitability
    GROUP BY amount_range
    ORDER BY amount_range
""").fetchdf()

# Sort the amount ranges in ascending order
amount_range_order = ['0-100', '101-500', '501-1000', '1001-2000', '2001-5000', '5000+']
profitability_by_amount['amount_range'] = pd.Categorical(
    profitability_by_amount['amount_range'],
    categories=amount_range_order,
    ordered=True
)
profitability_by_amount = profitability_by_amount.sort_values('amount_range')

# Visualize profitability by loan amount
fig9 = make_subplots(rows=1, cols=2,
                     subplot_titles=("Average Profit by Loan Amount", "Profit Margin by Loan Amount"),
                     specs=[[{"secondary_y": True}, {"secondary_y": False}]])

# Average profit by loan amount
fig9.add_trace(
    go.Bar(x=profitability_by_amount['amount_range'], y=profitability_by_amount['avg_profit'],
           name="Avg Profit", marker_color='mediumseagreen'),
    row=1, col=1
)

# Add loan count as line on secondary y-axis
fig9.add_trace(
    go.Scatter(x=profitability_by_amount['amount_range'], y=profitability_by_amount['loan_count'],
               name="Loan Count", mode='lines+markers', line=dict(color='darkblue', width=2)),
    row=1, col=1, secondary_y=True
)

# Profit margin by loan amount
fig9.add_trace(
    go.Bar(x=profitability_by_amount['amount_range'], y=profitability_by_amount['profit_margin_pct'],
           name="Profit Margin (%)", marker_color='darkorange'),
    row=1, col=2
)

# Update layout
fig9.update_layout(
    title_text="Profitability Analysis by Loan Amount",
    height=500,
    width=1200,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update axes titles
fig9.update_yaxes(title_text="Average Profit", row=1, col=1)
fig9.update_yaxes(title_text="Loan Count", row=1, col=1, secondary_y=True)
fig9.update_yaxes(title_text="Profit Margin (%)", row=1, col=2)
fig9.update_xaxes(title_text="Loan Amount Range", row=1, col=1)
fig9.update_xaxes(title_text="Loan Amount Range", row=1, col=2)

fig9.show()

# 4.3 Opportunity analysis: Tenure vs. profitability
profitability_by_tenure = con.execute("""
    WITH loan_profitability AS (
        SELECT
            d.customer_id,
            d.tenure_days,
            d.loan_amount,
            d.loan_fee,
            COALESCE(SUM(r.amount), 0) as total_repaid,
            CASE
                WHEN COALESCE(SUM(r.amount), 0) >= (d.loan_amount + d.loan_fee) THEN
                    COALESCE(SUM(r.amount), 0) - d.loan_amount
                ELSE
                    COALESCE(SUM(r.amount), 0) - d.loan_amount
            END as profit
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id  -- Join on customer_id
        WHERE d.expected_repayment_date::DATE <= CURRENT_DATE::DATE  -- Explicit date casting
        GROUP BY d.customer_id, d.tenure_days, d.loan_amount, d.loan_fee
    )
    SELECT
        tenure_days,
        COUNT(*) as loan_count,
        AVG(loan_amount) as avg_loan_amount,
        AVG(loan_fee) as avg_fee,
        AVG(profit) as avg_profit,
        SUM(profit) as total_profit,
        (AVG(profit) / AVG(loan_amount)) * 100 as profit_margin_pct,
        (AVG(profit) / tenure_days) as daily_profit
    FROM loan_profitability
    GROUP BY tenure_days
    ORDER BY tenure_days
""").fetchdf()

# Visualize profitability by tenure
fig10 = make_subplots(rows=1, cols=2,
                      subplot_titles=("Profit Metrics by Tenure", "Daily Profit by Tenure"),
                      specs=[[{"secondary_y": True}, {"secondary_y": False}]])

# Profit metrics by tenure
fig10.add_trace(
    go.Bar(x=profitability_by_tenure['tenure_days'].astype(str), y=profitability_by_tenure['avg_profit'],
           name="Avg Profit", marker_color='mediumseagreen'),
    row=1, col=1
)

# Add profit margin as line on secondary y-axis
fig10.add_trace(
    go.Scatter(x=profitability_by_tenure['tenure_days'].astype(str), y=profitability_by_tenure['profit_margin_pct'],
               name="Profit Margin (%)", mode='lines+markers', line=dict(color='darkblue', width=2)),
    row=1, col=1, secondary_y=True
)

# Daily profit by tenure
fig10.add_trace(
    go.Bar(x=profitability_by_tenure['tenure_days'].astype(str), y=profitability_by_tenure['daily_profit'],
           name="Daily Profit", marker_color='darkorange'),
    row=1, col=2
)

# Update layout
fig10.update_layout(
    title_text="Profitability Analysis by Tenure",
    height=500,
    width=1200,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update axes titles
fig10.update_yaxes(title_text="Average Profit", row=1, col=1)
fig10.update_yaxes(title_text="Profit Margin (%)", row=1, col=1, secondary_y=True)
fig10.update_yaxes(title_text="Daily Profit", row=1, col=2)
fig10.update_xaxes(title_text="Tenure (days)", row=1, col=1)
fig10.update_xaxes(title_text="Tenure (days)", row=1, col=2)

fig10.show()


# 4.4 Customer segment analysis
customer_segments = con.execute("""
    WITH loan_level_repayments AS (
        SELECT
            d.customer_id,
            d.loan_id,
            d.loan_amount,
            d.loan_fee,
            COALESCE(SUM(r.amount), 0) as total_repaid
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id AND d.loan_id = r.loan_id
        GROUP BY d.customer_id, d.loan_id, d.loan_amount, d.loan_fee
    ),
    customer_repayments AS (
        SELECT
            customer_id,
            COUNT(loan_id) as loans_with_repayments,
            SUM(total_repaid) as total_repaid,
            SUM(CASE WHEN total_repaid >= (loan_amount + loan_fee) THEN 1 ELSE 0 END) as fully_paid_loans
        FROM loan_level_repayments
        GROUP BY customer_id
    ),
    customer_loans AS (
        SELECT
            customer_id,
            COUNT(*) as loan_count,
            AVG(loan_amount) as avg_loan_amount,
            SUM(loan_amount) as total_borrowed,
            MIN(disb_date) as first_loan_date,
            MAX(disb_date) as last_loan_date,
            (MAX(disb_date) - MIN(disb_date)) as customer_lifetime_days
        FROM disbursements
        GROUP BY customer_id
    ),
    customer_segments AS (
        SELECT
            cl.customer_id,
            cl.loan_count,
            cl.avg_loan_amount,
            cl.total_borrowed,
            cl.first_loan_date,
            cl.last_loan_date,
            cl.customer_lifetime_days,
            COALESCE(cr.loans_with_repayments, 0) as loans_with_repayments,
            COALESCE(cr.total_repaid, 0) as total_repaid,
            COALESCE(cr.fully_paid_loans, 0) as fully_paid_loans,
            CASE
                WHEN cl.loan_count > 0 THEN (COALESCE(cr.fully_paid_loans, 0) * 100.0 / cl.loan_count)
                ELSE 0  -- Handle the case where loan_count is zero
            END as repayment_rate,
            CASE
                WHEN cl.loan_count = 1 THEN 'One-time'
                WHEN cl.loan_count <= 3 THEN 'Occasional'
                WHEN cl.loan_count <= 10 THEN 'Regular'
                ELSE 'Power User'
            END as frequency_segment,
            CASE
                WHEN cl.avg_loan_amount <= 100 THEN 'Micro'
                WHEN cl.avg_loan_amount <= 500 THEN 'Small'
                WHEN cl.avg_loan_amount <= 2000 THEN 'Medium'
                ELSE 'Large'
            END as amount_segment,
            CASE
                WHEN (COALESCE(cr.fully_paid_loans, 0) * 100.0 / cl.loan_count) >= 95 THEN 'Excellent'
                WHEN (COALESCE(cr.fully_paid_loans, 0) * 100.0 / cl.loan_count) >= 80 THEN 'Good'
                WHEN (COALESCE(cr.fully_paid_loans, 0) * 100.0 / cl.loan_count) >= 50 THEN 'Fair'
                ELSE 'Poor'
            END as repayment_segment
        FROM customer_loans cl
        LEFT JOIN customer_repayments cr ON cl.customer_id = cr.customer_id
    )
    SELECT
        frequency_segment,
        amount_segment,
        repayment_segment,
        COUNT(*) as customer_count,
        AVG(loan_count) as avg_loans_per_customer,
        AVG(avg_loan_amount) as avg_loan_amount,
        AVG(repayment_rate) as avg_repayment_rate,
        SUM(total_borrowed) as total_borrowed,
        SUM(total_repaid) as total_repaid
    FROM customer_segments
    GROUP BY frequency_segment, amount_segment, repayment_segment
    ORDER BY frequency_segment, amount_segment, repayment_segment
""").fetchdf()

# Create a summary of customer segments
segment_summary = con.execute("""
    WITH loan_level_repayments AS (
        SELECT
            d.customer_id,
            d.loan_id,
            d.loan_amount,
            d.loan_fee,
            COALESCE(SUM(r.amount), 0) as total_repaid
        FROM disbursements d
        LEFT JOIN repayments r ON d.customer_id = r.customer_id AND d.loan_id = r.loan_id
        GROUP BY d.customer_id, d.loan_id, d.loan_amount, d.loan_fee
    ),
    customer_repayments AS (
        SELECT
            customer_id,
            COUNT(loan_id) as loans_with_repayments,
            SUM(total_repaid) as total_repaid,
            SUM(CASE WHEN total_repaid >= (loan_amount + loan_fee) THEN 1 ELSE 0 END) as fully_paid_loans
        FROM loan_level_repayments
        GROUP BY customer_id
    ),
    customer_loans AS (
        SELECT
            customer_id,
            COUNT(*) as loan_count,
            AVG(loan_amount) as avg_loan_amount,
            SUM(loan_amount) as total_borrowed,
            MIN(disb_date) as first_loan_date,
            MAX(disb_date) as last_loan_date,
            (MAX(disb_date) - MIN(disb_date)) as customer_lifetime_days
        FROM disbursements
        GROUP BY customer_id
    ),
    customer_segments AS (
        SELECT
            cl.customer_id,
            cl.loan_count,
            cl.avg_loan_amount,
            cl.total_borrowed,
            cl.first_loan_date,
            cl.last_loan_date,
            cl.customer_lifetime_days,
            COALESCE(cr.loans_with_repayments, 0) as loans_with_repayments,
            COALESCE(cr.total_repaid, 0) as total_repaid,
            COALESCE(cr.fully_paid_loans, 0) as fully_paid_loans,
            CASE
                WHEN cl.loan_count > 0 THEN (COALESCE(cr.fully_paid_loans, 0) * 100.0 / cl.loan_count)
                ELSE 0  -- Handle the case where loan_count is zero
            END as repayment_rate,
            CASE
                WHEN cl.loan_count = 1 THEN 'One-time'
                WHEN cl.loan_count <= 3 THEN 'Occasional'
                WHEN cl.loan_count <= 10 THEN 'Regular'
                ELSE 'Power User'
            END as frequency_segment
        FROM customer_loans cl
        LEFT JOIN customer_repayments cr ON cl.customer_id = cr.customer_id
    )
    SELECT
        frequency_segment,
        COUNT(*) as customer_count,
        AVG(loan_count) as avg_loans_per_customer,
        AVG(avg_loan_amount) as avg_loan_amount,
        AVG(repayment_rate) as avg_repayment_rate, -- Re-add AVG(repayment_rate)
        SUM(total_borrowed) as total_borrowed,
        SUM(total_repaid) as total_repaid,
        (SUM(total_repaid) - SUM(total_borrowed)) as total_profit
    FROM customer_segments
    GROUP BY frequency_segment
    ORDER BY
        CASE
            WHEN frequency_segment = 'One-time' THEN 1
            WHEN frequency_segment = 'Occasional' THEN 2
            WHEN frequency_segment = 'Regular' THEN 3
            WHEN frequency_segment = 'Power User' THEN 4
        END
""").fetchdf()

# Visualize customer segment profitability
fig11 = make_subplots(rows=1, cols=2,
                      subplot_titles=("Customer Segment Metrics", "Profitability by Customer Segment"),
                      specs=[[{"secondary_y": True}, {"secondary_y": True}]])

# Customer segment metrics
fig11.add_trace(
    go.Bar(x=segment_summary['frequency_segment'], y=segment_summary['customer_count'],
           name="Customer Count", marker_color='royalblue'),
    row=1, col=1
)

# Add average loans per customer as line on secondary y-axis
fig11.add_trace(
    go.Scatter(x=segment_summary['frequency_segment'], y=segment_summary['avg_loans_per_customer'],
           name="Avg Loans per Customer", mode='lines+markers', line=dict(color='firebrick', width=2)),
    row=1, col=1, secondary_y=True
)

# Profitability by customer segment
fig11.add_trace(
    go.Bar(x=segment_summary['frequency_segment'], y=segment_summary['total_profit'],
           name="Total Profit", marker_color='mediumseagreen'),
    row=1, col=2
)

# Add repayment rate as line on secondary y-axis
fig11.add_trace(
    go.Scatter(x=segment_summary['frequency_segment'], y=segment_summary['avg_repayment_rate'],
           name="Avg Repayment Rate (%)", mode='lines+markers', line=dict(color='darkorange', width=2)),
    row=1, col=2, secondary_y=True
)

# Update layout
fig11.update_layout(
    title_text="Customer Segment Analysis",
    height=500,
    width=1200,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update axes titles
fig11.update_yaxes(title_text="Customer Count", row=1, col=1)
fig11.update_yaxes(title_text="Avg Loans per Customer", row=1, col=1, secondary_y=True)
fig11.update_yaxes(title_text="Total Profit", row=1, col=2)
fig11.update_yaxes(title_text="Repayment Rate (%)", row=1, col=2, secondary_y=True)
fig11.update_xaxes(title_text="Customer Segment", row=1, col=1)
fig11.update_xaxes(title_text="Customer Segment", row=1, col=2)

fig11.show()

# 4.5 Generate summary of findings
print("\n=== SUMMARY OF KEY FINDINGS ===")
print("\n1. Loan Volume and Revenue Trends:")
print(f"- Total loans: {disbursements_clean.shape[0]:,}")
print(f"- Total loan amount: {disbursements_clean['loan_amount'].sum():,.2f}")
print(f"- Total fee revenue: {disbursements_clean['loan_fee'].sum():,.2f}")
print(f"- Average loan amount: {disbursements_clean['loan_amount'].mean():,.2f}")
print(f"- Average fee: {disbursements_clean['loan_fee'].mean():,.2f}")

print("\n2. Tenure Analysis:")
for tenure in sorted(disbursements_clean['tenure_days'].unique()):
    tenure_data = disbursements_clean[disbursements_clean['tenure_days'] == tenure]
    print(f"- {tenure}-day loans: {tenure_data.shape[0]:,} loans ({tenure_data.shape[0]/disbursements_clean.shape[0]*100:.1f}%)")
    print(f"  Average amount: {tenure_data['loan_amount'].mean():,.2f}")
    print(f"  Average fee: {tenure_data['loan_fee'].mean():,.2f} ({tenure_data['loan_fee'].mean()/tenure_data['loan_amount'].mean()*100:.1f}%)")

print("\n3. Repayment Performance:")
if 'repayment_type' in repayments_clean.columns:
    for repayment_type, data in repayment_rates.iterrows():
        print(f"- {data['repayment_type']} repayments: {data['repayment_rate_by_amount']:.1f}% repayment rate")

print("\n4. Default Analysis:")
print(f"- Overall default rate: {default_rates['default_rate'].mean():.2f}%")
print(f"- Highest monthly default rate: {default_rates['default_rate'].max():.2f}% ({default_rates.loc[default_rates['default_rate'].idxmax(), 'month']})")
print(f"- Lowest monthly default rate: {default_rates['default_rate'].min():.2f}% ({default_rates.loc[default_rates['default_rate'].idxmin(), 'month']})")

print("\n5. Profitability Analysis:")
print(f"- Most profitable loan amount range: {profitability_by_amount.loc[profitability_by_amount['avg_profit'].idxmax(), 'amount_range']} (Avg profit: {profitability_by_amount['avg_profit'].max():.2f})")
print(f"- Most profitable tenure: {profitability_by_tenure.loc[profitability_by_tenure['avg_profit'].idxmax(), 'tenure_days']} days (Avg profit: {profitability_by_tenure['avg_profit'].max():.2f})")
print(f"- Highest daily profit tenure: {profitability_by_tenure.loc[profitability_by_tenure['daily_profit'].idxmax(), 'tenure_days']} days (Daily profit: {profitability_by_tenure['daily_profit'].max():.2f})")

print("\n6. Customer Segment Analysis:")
print(f"- Most profitable customer segment: {segment_summary.loc[segment_summary['total_profit'].idxmax(), 'frequency_segment']} (Total profit: {segment_summary['total_profit'].max():.2f})")
print(f"- Highest repayment rate segment: {segment_summary.loc[segment_summary['avg_repayment_rate'].idxmax(), 'frequency_segment']} ({segment_summary['avg_repayment_rate'].max():.2f}%)")
print(f"- Largest customer segment: {segment_summary.loc[segment_summary['customer_count'].idxmax(), 'frequency_segment']} ({segment_summary['customer_count'].max():,} customers, {segment_summary['customer_count'].max()/segment_summary['customer_count'].sum()*100:.1f}%)")

print("\n7. Key Correlations:")
print(f"- Loan amount and default probability: {amount_default_corr:.4f}")
print(f"- Tenure and repayment percentage: {tenure_repayment_corr:.4f}")

print("\n=== CONCERNING TRENDS ===")
# Identify concerning trends based on the analysis
concerning_months_count = default_trend['is_concerning'].sum()
if concerning_months_count > 0:
    print(f"- Found {concerning_months_count} months with concerning default rates above the threshold")
    for _, month in concerning_months[['month', 'default_rate']].iterrows():
        print(f"  * {month['month']}: {month['default_rate']:.2f}% default rate")

# Check for concerning correlations
if amount_default_corr > 0.2:
    print(f"- Positive correlation between loan amount and default probability ({amount_default_corr:.4f})")
    print("  This suggests higher loan amounts have higher default risk")
elif amount_default_corr < -0.2:
    print(f"- Negative correlation between loan amount and default probability ({amount_default_corr:.4f})")
    print("  This suggests lower loan amounts have higher default risk")

if tenure_repayment_corr < -0.2:
    print(f"- Negative correlation between tenure and repayment percentage ({tenure_repayment_corr:.4f})")
    print("  This suggests longer tenures have lower repayment rates")

# Check for concerning profitability trends
negative_profit_amounts = profitability_by_amount[profitability_by_amount['avg_profit'] < 0]
if not negative_profit_amounts.empty:
    print("- Negative profitability in the following loan amount ranges:")
    for _, row in negative_profit_amounts.iterrows():
        print(f"  * {row['amount_range']}: {row['avg_profit']:.2f} average profit, {row['profit_margin_pct']:.2f}% margin")

negative_profit_tenures = profitability_by_tenure[profitability_by_tenure['avg_profit'] < 0]
if not negative_profit_tenures.empty:
    print("- Negative profitability in the following tenure options:")
    for _, row in negative_profit_tenures.iterrows():
        print(f"  * {row['tenure_days']} days: {row['avg_profit']:.2f} average profit, {row['profit_margin_pct']:.2f}% margin")

print("\n=== OPPORTUNITIES ===")
# Identify opportunities based on the analysis
# Most profitable segments
print("1. Optimize for most profitable segments:")
print(f"- Focus on {profitability_by_amount.loc[profitability_by_amount['profit_margin_pct'].idxmax(), 'amount_range']} loan amounts (Profit margin: {profitability_by_amount['profit_margin_pct'].max():.2f}%)")
print(f"- Promote {profitability_by_tenure.loc[profitability_by_tenure['profit_margin_pct'].idxmax(), 'tenure_days']} days (Profit margin: {profitability_by_tenure['profit_margin_pct'].max():.2f}%)")
print(f"- Target {segment_summary.loc[segment_summary['avg_repayment_rate'].idxmax(), 'frequency_segment']} customers (Repayment rate: {segment_summary['avg_repayment_rate'].max():.2f}%)")

# Underserved segments with good performance
high_performing_segments = customer_segments[customer_segments['avg_repayment_rate'] > 80]
if not high_performing_segments.empty:
    underserved = high_performing_segments[customer_segments['customer_count'] < customer_segments['customer_count'].median()]
    if not underserved.empty:
        print("\n2. Expand into underserved segments with good performance:")
        for _, row in underserved.head(3).iterrows():
            print(f"- {row['frequency_segment']} {row['amount_segment']} borrowers with {row['repayment_segment']} repayment history")
            print(f"  Only {row['customer_count']} customers but {row['avg_repayment_rate']:.2f}% repayment rate")

# Seasonal opportunities
monthly_volumes['month_name'] = monthly_volumes['month'].dt.strftime('%B')
low_volume_months = monthly_volumes[monthly_volumes['loan_count'] < monthly_volumes['loan_count'].median()]
if not low_volume_months.empty:
    print("\n3. Seasonal marketing opportunities:")
    for _, row in low_volume_months.iterrows():
        print(f"- {row['month_name']}: Only {row['loan_count']} loans vs. {monthly_volumes['loan_count'].median():.0f} median")
        print(f"  Potential for targeted promotions to increase volume")

# Product enhancement opportunities
print("\n4. Product enhancement opportunities:")
if 'repayment_type' in repayments_clean.columns:
    auto_repayment_rate = repayment_rates[repayment_rates['repayment_type'] == 'Automatic']['repayment_rate_by_amount'].values[0]
    manual_repayment_rate = repayment_rates[repayment_rates['repayment_type'] == 'Manual']['repayment_rate_by_amount'].values[0]

    if auto_repayment_rate > manual_repayment_rate:
        print(f"- Encourage automatic repayments: {auto_repayment_rate:.1f}% vs {manual_repayment_rate:.1f}% repayment rate")
        print("  Consider incentives or discounts for customers who opt for automatic repayments")

# Check for gaps in tenure options
existing_tenures = sorted(disbursements_clean['tenure_days'].unique())
if len(existing_tenures) < 5:  # Assuming there could be more tenure options
    print("- Consider introducing additional tenure options:")
    potential_tenures = [t for t in [1, 3, 7, 14, 21, 30, 45, 60, 90] if t not in existing_tenures]
    for tenure in potential_tenures[:3]:  # Suggest up to 3 new tenures
        print(f"  * {tenure}-day loans could fill gaps in current offerings")

# Check for gaps in loan amount ranges
loan_amount_distribution = pd.cut(disbursements_clean['loan_amount'],
                                 bins=[0, 100, 500, 1000, 2000, 5000, float('inf')],
                                 labels=['0-100', '101-500', '501-1000', '1001-2000', '2001-5000', '5000+'])
amount_counts = loan_amount_distribution.value_counts(normalize=True) * 100

low_volume_ranges = amount_counts[amount_counts < 10]  # Ranges with less than 10% of loans
if not low_volume_ranges.empty:
    print("- Potential to expand into underserved loan amount ranges:")
    for range_name, percentage in low_volume_ranges.items():
        print(f"  * {range_name}: Only {percentage:.1f}% of current loans")

# Check for customer retention opportunities
if 'customer_lifetime_days' in customer_segments.columns and not customer_segments['customer_lifetime_days'].isnull().all():
    retention_opportunity = con.execute("""
        WITH customer_loans AS (
            SELECT
                customer_id,
                COUNT(*) as loan_count,
                MIN(disb_date) as first_loan_date,
                MAX(disb_date) as last_loan_date,
                (MAX(disb_date) - MIN(disb_date)) as customer_lifetime_days
            FROM disbursements
            GROUP BY customer_id
        )
        SELECT
            AVG(customer_lifetime_days) as avg_lifetime_days,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY customer_lifetime_days) as median_lifetime_days,
            COUNT(*) as total_customers,
            COUNT(CASE WHEN loan_count = 1 THEN 1 END) as one_time_customers,
            (COUNT(CASE WHEN loan_count = 1 THEN 1 END) * 100.0 / COUNT(*)) as one_time_percentage
        FROM customer_loans
    """).fetchdf()

    print("\n5. Customer retention opportunities:")
    print(f"- {retention_opportunity['one_time_percentage'].values[0]:.1f}% of customers ({retention_opportunity['one_time_customers'].values[0]:,}) only took one loan")
    print("  Implement a re-engagement strategy for one-time borrowers")
    print(f"- Average customer lifetime: {retention_opportunity['avg_lifetime_days'].values[0]:.1f} days")
    print("  Develop loyalty programs to extend customer lifetime and increase repeat borrowing")

# Create a comprehensive findings document
from IPython.display import Markdown, display

findings_document = """
# Performance Metrics and Time-Series Analysis Findings

## Executive Summary
This analysis examines the performance metrics and time-series trends of the lending product, focusing on loan volumes, revenue, customer behavior, and default patterns. The findings reveal both concerning trends that require attention and opportunities for product enhancement and business growth.

## Key Performance Indicators

### Loan Volume and Revenue
- Total loans: {total_loans:,}
- Total loan amount: {total_amount:,.2f}
- Total fee revenue: {total_fees:,.2f}
- Average loan amount: {avg_loan_amount:,.2f}
- Average fee: {avg_fee:,.2f} ({fee_percentage:.1f}% of loan amount)

### Tenure Distribution
{tenure_distribution}

### Repayment Performance
- Overall repayment rate: {overall_repayment_rate:.1f}%
- Repayment rate by type:
{repayment_by_type}

### Default Analysis
- Overall default rate: {overall_default_rate:.2f}%
- Default rate range: {min_default_rate:.2f}% to {max_default_rate:.2f}%
- Default rate by tenure:
{default_by_tenure_text}

### Customer Segments
- Total unique customers: {unique_customers:,}
- Customer segments by frequency:
{customer_segments_text}

## Time-Series Trends

### Loan Disbursement Trends
- {disbursement_trend_description}
- Highest volume month: {highest_volume_month} with {highest_volume_count:,} loans
- Lowest volume month: {lowest_volume_month} with {lowest_volume_count:,} loans

### Fee Revenue Trends
- {revenue_trend_description}
- Fee revenue shows {fee_correlation_with_volume_description} with loan volume

### Default Rate Evolution
- {default_trend_description}
- {concerning_months_count} months with concerning default rates above threshold

## Correlation Analysis

### Loan Amount and Default Probability
- Correlation coefficient: {amount_default_corr:.4f}
- {amount_default_interpretation}

### Tenure and Repayment Behavior
- Correlation coefficient: {tenure_repayment_corr:.4f}
- {tenure_repayment_interpretation}

### Repayment Type and Success Rate
- {repayment_type_correlation_description}

## Concerning Trends
{concerning_trends}

## Opportunities
{opportunities}

## Recommendations
1. **Optimize Product Mix**: Focus on the most profitable segments identified in the analysis
   - Promote {best_tenure}-day loans with amounts in the {best_amount_range} range
   - Target {best_customer_segment} customers with tailored marketing

2. **Enhance Risk Management**: Address the identified concerning trends
   - Implement stricter underwriting for high-risk segments
   - Consider adjusting fee structure to better account for risk

3. **Improve Customer Retention**: Develop strategies to convert one-time borrowers into repeat customers
   - Implement a loyalty program with incentives for repeat borrowing
   - Create a re-engagement campaign for dormant customers

4. **Product Enhancements**: Consider introducing new features based on the analysis
   - Add new tenure options to fill gaps in current offerings
   - Expand into underserved loan amount ranges with good profitability potential
   - Encourage automatic repayments through incentives or discounts

5. **Seasonal Strategy**: Develop targeted marketing campaigns for low-volume periods
   - Increase marketing efforts during {low_volume_months}
   - Consider seasonal promotions to boost loan volume during slower periods
"""

# Prepare values for the findings document
values = {
    'total_loans': disbursements_clean.shape[0],
    'total_amount': disbursements_clean['loan_amount'].sum(),
    'total_fees': disbursements_clean['loan_fee'].sum(),
    'avg_loan_amount': disbursements_clean['loan_amount'].mean(),
    'avg_fee': disbursements_clean['loan_fee'].mean(),
    'fee_percentage': (disbursements_clean['loan_fee'].sum() / disbursements_clean['loan_amount'].sum()) * 100,

    # Tenure distribution
    'tenure_distribution': '\n'.join([f"- {tenure}-day loans: {disbursements_clean[disbursements_clean['tenure_days'] == tenure].shape[0]:,} loans ({disbursements_clean[disbursements_clean['tenure_days'] == tenure].shape[0]/disbursements_clean.shape[0]*100:.1f}%)"
                                     for tenure in sorted(disbursements_clean['tenure_days'].unique())]),

    # Repayment performance
    'overall_repayment_rate': default_rates['default_rate'].mean(),
    'repayment_by_type': '\n'.join([f"  - {row['repayment_type']}: {row['repayment_rate_by_amount']:.1f}%"
                                   for _, row in repayment_rates.iterrows()]) if 'repayment_type' in repayments_clean.columns
                          else "  - Data not available",

    # Default analysis
    'overall_default_rate': default_rates['default_rate'].mean(),
    'min_default_rate': default_rates['default_rate'].min(),
    'max_default_rate': default_rates['default_rate'].max(),
    'default_by_tenure_text': '\n'.join([f"  - {row['tenure_days']}-day loans: {row['default_rate']:.2f}%"
                                        for _, row in default_by_tenure.iterrows()]),

    # Customer segments
    'unique_customers': len(disbursements_clean['customer_id'].unique()),
    'customer_segments_text': '\n'.join([f"  - {row['frequency_segment']}: {row['customer_count']:,} customers ({row['customer_count']/segment_summary['customer_count'].sum()*100:.1f}%)"
                                        for _, row in segment_summary.iterrows()]),

    # Time-series trends
    'disbursement_trend_description': "Loan disbursements show a [increasing/stable/decreasing] trend over time" +
                                     (" with seasonal patterns" if monthly_volumes.shape[0] >= 12 else ""),
    'highest_volume_month': monthly_volumes.loc[monthly_volumes['loan_count'].idxmax(), 'month'].strftime('%B %Y'),
    'highest_volume_count': monthly_volumes['loan_count'].max(),
    'lowest_volume_month': monthly_volumes.loc[monthly_volumes['loan_count'].idxmin(), 'month'].strftime('%B %Y'),
    'lowest_volume_count': monthly_volumes['loan_count'].min(),

    'revenue_trend_description': "Fee revenue shows a [increasing/stable/decreasing] trend over time",
    'fee_correlation_with_volume_description': "strong correlation" if monthly_volumes['loan_count'].corr(monthly_volumes['total_fees']) > 0.8 else "moderate correlation" if monthly_volumes['loan_count'].corr(monthly_volumes['total_fees']) > 0.5 else "weak correlation",

    # Default_trend_description
    'default_trend_description': "Default rates show a [increasing/stable/decreasing] trend over time",
    'concerning_months_count': default_trend['is_concerning'].sum(),

    # Correlation analysis
    'amount_default_corr': amount_default_corr,
    'amount_default_interpretation': "Higher loan amounts are associated with higher default probability" if amount_default_corr > 0.2 else
                                    "Lower loan amounts are associated with higher default probability" if amount_default_corr < -0.2 else
                                    "No strong relationship between loan amount and default probability",

    'tenure_repayment_corr': tenure_repayment_corr,
    'tenure_repayment_interpretation': "Longer tenures are associated with lower repayment rates" if tenure_repayment_corr < -0.2 else
                                      "Longer tenures are associated with higher repayment rates" if tenure_repayment_corr > 0.2 else
                                      "No strong relationship between tenure and repayment behavior",

    'repayment_type_correlation_description': "Automatic repayments show higher success rates than manual repayments"
                                             if 'repayment_type' in repayments_clean.columns and
                                                repayment_rates[repayment_rates['repayment_type'] == 'Automatic']['repayment_rate_by_amount'].values[0] >
                                                repayment_rates[repayment_rates['repayment_type'] == 'Manual']['repayment_rate_by_amount'].values[0]
                                             else "Manual repayments show higher success rates than automatic repayments"
                                             if 'repayment_type' in repayments_clean.columns else
                                             "Data not available for analysis",

    # Concerning trends
    'concerning_trends': "1. Default rates show concerning increases in certain periods\n" +
                        "2. Some loan amount ranges show negative profitability\n" +
                        "3. High percentage of one-time borrowers indicates retention challenges" if 'customer_lifetime_days' in customer_segments.columns else
                        "3. Customer retention data not available for analysis",

    # Opportunities
    'opportunities': "1. Focus on most profitable customer segments and loan parameters\n" +
                    "2. Introduce new tenure options to fill gaps in current offerings\n" +
                    "3. Implement targeted marketing during low-volume periods\n" +
                    "4. Enhance automatic repayment adoption through incentives\n" +
                    "5. Develop loyalty programs to improve customer retention",

    # Recommendations
    'best_tenure': profitability_by_tenure.loc[profitability_by_tenure['profit_margin_pct'].idxmax(), 'tenure_days'],
    'best_amount_range': profitability_by_amount.loc[profitability_by_amount['profit_margin_pct'].idxmax(), 'amount_range'],
    'best_customer_segment': segment_summary.loc[segment_summary['avg_repayment_rate'].idxmax(), 'frequency_segment'],
    'low_volume_months': ', '.join(low_volume_months['month_name'].unique()[:3]) if not low_volume_months.empty else "identified low-volume periods"
}

# Display the findings document
display(Markdown(findings_document.format(**values)))

# Save the findings to a file
with open('performance_metrics_findings.md', 'w') as f:
    f.write(findings_document.format(**values))

print("\nAnalysis complete. Findings document saved as 'performance_metrics_findings.md'")


4. Identifying concerning trends and opportunities...



=== SUMMARY OF KEY FINDINGS ===

1. Loan Volume and Revenue Trends:
- Total loans: 26,585
- Total loan amount: 26,612,154.00
- Total fee revenue: 3,448,045.36
- Average loan amount: 1,001.02
- Average fee: 129.70

2. Tenure Analysis:
- 7-day loans: 7,839 loans (29.5%)
  Average amount: 794.98
  Average fee: 79.50 (10.0%)
- 14-day loans: 12,948 loans (48.7%)
  Average amount: 597.74
  Average fee: 71.73 (12.0%)
- 30-day loans: 5,798 loans (21.8%)
  Average amount: 2,180.20
  Average fee: 327.03 (15.0%)

3. Repayment Performance:
- Automatic repayments: 46.0% repayment rate
- Manual repayments: 59.5% repayment rate

4. Default Analysis:
- Overall default rate: 0.14%
- Highest monthly default rate: 0.28% (2024-01-01 00:00:00)
- Lowest monthly default rate: 0.05% (2024-05-01 00:00:00)

5. Profitability Analysis:
- Most profitable loan amount range: 2001-5000 (Avg profit: 120723.28)
- Most profitable tenure: 30 days (Avg profit: 41785.73)
- Highest daily profit tenure: 7 days (Daily profit


# Performance Metrics and Time-Series Analysis Findings

## Executive Summary
This analysis examines the performance metrics and time-series trends of the lending product, focusing on loan volumes, revenue, customer behavior, and default patterns. The findings reveal both concerning trends that require attention and opportunities for product enhancement and business growth.

## Key Performance Indicators

### Loan Volume and Revenue
- Total loans: 26,585
- Total loan amount: 26,612,154.00
- Total fee revenue: 3,448,045.36
- Average loan amount: 1,001.02
- Average fee: 129.70 (13.0% of loan amount)

### Tenure Distribution
- 7-day loans: 7,839 loans (29.5%)
- 14-day loans: 12,948 loans (48.7%)
- 30-day loans: 5,798 loans (21.8%)

### Repayment Performance
- Overall repayment rate: 0.1%
- Repayment rate by type:
  - Automatic: 46.0%
  - Manual: 59.5%

### Default Analysis
- Overall default rate: 0.14%
- Default rate range: 0.05% to 0.28%
- Default rate by tenure:
  - 7.0-day loans: 0.17%
  - 14.0-day loans: 0.12%
  - 30.0-day loans: 0.17%

### Customer Segments
- Total unique customers: 2,996
- Customer segments by frequency:
  - One-time: 77 customers (2.6%)
  - Occasional: 293 customers (9.8%)
  - Regular: 1,776 customers (59.3%)
  - Power User: 850 customers (28.4%)

## Time-Series Trends

### Loan Disbursement Trends
- Loan disbursements show a [increasing/stable/decreasing] trend over time
- Highest volume month: March 2024 with 4,057 loans
- Lowest volume month: August 2024 with 1,139 loans

### Fee Revenue Trends
- Fee revenue shows a [increasing/stable/decreasing] trend over time
- Fee revenue shows strong correlation with loan volume

### Default Rate Evolution
- Default rates show a [increasing/stable/decreasing] trend over time
- 1 months with concerning default rates above threshold

## Correlation Analysis

### Loan Amount and Default Probability
- Correlation coefficient: -0.0196
- No strong relationship between loan amount and default probability

### Tenure and Repayment Behavior
- Correlation coefficient: -0.1683
- No strong relationship between tenure and repayment behavior

### Repayment Type and Success Rate
- Manual repayments show higher success rates than automatic repayments

## Concerning Trends
3. Customer retention data not available for analysis

## Opportunities
1. Focus on most profitable customer segments and loan parameters
2. Introduce new tenure options to fill gaps in current offerings
3. Implement targeted marketing during low-volume periods
4. Enhance automatic repayment adoption through incentives
5. Develop loyalty programs to improve customer retention

## Recommendations
1. **Optimize Product Mix**: Focus on the most profitable segments identified in the analysis
   - Promote 7-day loans with amounts in the 0-100 range
   - Target One-time customers with tailored marketing

2. **Enhance Risk Management**: Address the identified concerning trends
   - Implement stricter underwriting for high-risk segments
   - Consider adjusting fee structure to better account for risk

3. **Improve Customer Retention**: Develop strategies to convert one-time borrowers into repeat customers
   - Implement a loyalty program with incentives for repeat borrowing
   - Create a re-engagement campaign for dormant customers

4. **Product Enhancements**: Consider introducing new features based on the analysis
   - Add new tenure options to fill gaps in current offerings
   - Expand into underserved loan amount ranges with good profitability potential
   - Encourage automatic repayments through incentives or discounts

5. **Seasonal Strategy**: Develop targeted marketing campaigns for low-volume periods
   - Increase marketing efforts during January, June, July
   - Consider seasonal promotions to boost loan volume during slower periods



Analysis complete. Findings document saved as 'performance_metrics_findings.md'
